## Feature Engineering Pipeline - Heart Disease Dataset

### By:
Laura Granda

### Date:
2026-03-15

### Description:
Feature engineering pipeline for heart disease classification.
Selects 9 features, removes duplicates, splits into train/test,
and creates a ColumnTransformer preprocessor with numeric, categorical,
and ordinal pipelines.

Input:  `data/03_primary/corazon_explored.parquet`
Output: Transformed training data with engineered features

## 📚 Import libraries

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

## 💾 Load data

In [ ]:
INPUT_PATH: Path = Path("/home/lauragranda01/corazon/data/03_primary/corazon_explored.parquet")
df: pd.DataFrame = pd.read_parquet(INPUT_PATH, engine="pyarrow")
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (2919, 14)


,age,sex,chest_pain,rest_bp,chol,fbs,rest_ecg,max_hr,exang,old_peak,slope,ca,thal,disease
0,63,Male,typical,145,233,True,left ventricular hypertrophy,150,False,2.3,3,0.0,fixed,False
1,67,Male,asymptomatic,160,286,False,left ventricular hypertrophy,108,True,1.5,2,3.0,normal,True
2,67,Male,asymptomatic,120,229,False,left ventricular hypertrophy,129,True,2.6,2,2.0,reversable,True
3,37,Male,nonanginal,130,250,False,normal,187,False,3.5,3,0.0,normal,False
4,41,Female,nontypical,130,204,False,left ventricular hypertrophy,172,False,1.4,1,0.0,normal,False


## 👷 Data Preparation

### Select features

In [ ]:
TARGET: str = "disease"
SELECTED_COLUMNS: list[str] = [
    "age",
    "max_hr",
    "old_peak",
    "chest_pain",
    "thal",
    "exang",
    "slope",
    "ca",
    "sex",
    TARGET,
]
df_selected: pd.DataFrame = df[SELECTED_COLUMNS].copy()
print(f"Selected features shape: {df_selected.shape}")
df_selected.info()

Selected features shape: (2919, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2919 entries, 0 to 2918
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   age         2902 non-null   Int64   
 1   max_hr      2829 non-null   Int64   
 2   old_peak    2849 non-null   float64 
 3   chest_pain  2848 non-null   category
 4   thal        2872 non-null   category
 5   exang       2848 non-null   boolean 
 6   slope       2849 non-null   category
 7   ca          2838 non-null   category
 8   sex         2870 non-null   category
 9   disease     2919 non-null   boolean 
dtypes: Int64(2), boolean(2), category(5), float64(1)
memory usage: 100.7 KB


### Check missing values

In [4]:
missing_values: pd.Series = df_selected.isnull().sum()
print("Missing values per column:")
print(missing_values[missing_values > 0])

Missing values per column:
age           17
max_hr        90
old_peak      70
chest_pain    71
thal          47
exang         71
slope         70
ca            81
sex           49
dtype: int64


### Check duplicates

In [5]:
n_duplicates: int = df_selected.duplicated().sum()
print(f"Number of duplicate rows: {n_duplicates}")

Number of duplicate rows: 2463


### Remove duplicates

In [ ]:
df_clean: pd.DataFrame = df_selected.drop_duplicates().reset_index(drop=True)
n_removed: int = len(df_selected) - len(df_clean)
print(f"Removed {n_removed} duplicate rows")
print(f"Dataset shape after deduplication: {df_clean.shape}")

# Convert categorical and boolean columns to string dtype for sklearn compatibility
# This handles both the type conversion and NA values appropriately
for col in df_clean.columns:
    if df_clean[col].dtype in ("category", "boolean", "bool"):
        df_clean[col] = df_clean[col].astype(str)

Removed 2463 duplicate rows
Dataset shape after deduplication: (456, 10)


## 🔧 Feature Engineering

### Define feature groups

In [7]:
NUMERIC_FEATURES: list[str] = ["age", "max_hr", "old_peak"]
CATEGORICAL_FEATURES: list[str] = ["chest_pain", "sex"]
ORDINAL_FEATURES: list[str] = ["thal", "slope", "ca", "exang"]

print("Numeric features:", NUMERIC_FEATURES)
print("Categorical features:", CATEGORICAL_FEATURES)
print("Ordinal features:", ORDINAL_FEATURES)

Numeric features: ['age', 'max_hr', 'old_peak']
Categorical features: ['chest_pain', 'sex']
Ordinal features: ['thal', 'slope', 'ca', 'exang']


### Create pipelines

In [ ]:
# Numeric pipeline
numeric_transformer: Pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

# Categorical pipeline
categorical_transformer: Pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# Ordinal pipeline
ordinal_transformer: Pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        ),
    ]
)

print("Pipelines created successfully")

Pipelines created successfully


### Combine into ColumnTransformer

In [9]:
preprocessor: ColumnTransformer = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
        ("ord", ordinal_transformer, ORDINAL_FEATURES),
    ]
)

print("ColumnTransformer created:")
print(preprocessor)

ColumnTransformer created:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['age', 'max_hr', 'old_peak']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['chest_pain', 'sex']),
                                ('ord',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                       

## 📊 Results

### Feature Engineering Summary

**Data Preparation:**
- Selected 9 features plus target variable from 14 original columns
- Identified and removed duplicate rows
- Final clean dataset: 2919 rows × 10 columns (9 features + target)

**Feature Engineering:**
- Numerical features (3): age, max_hr, old_peak → Median imputation
- Categorical features (2): chest_pain, sex → Mode imputation + One-Hot Encoding
- Ordinal features (4): thal, slope, ca, exang → Mode imputation + Ordinal Encoding

